# Tutorial 1 -- Data sources

The v3 paradigm in cfdmod is built around one value object: the **DataSource**. Every recipe (Cp, Cf, S1, dynamic) starts with one and ends with one.

A `DataSource` bundles five things:

| | Purpose |
|---|---|
| `time` | affine `TimeAxis(initial_time, timestep_size, n_timesteps)` -- never a materialised array |
| `topology` | mesh connectivity + vertices (or just bare points) |
| `elements` | per-element scalars/vectors: position, area, volume, normal |
| `groupings` | dict of named partitions over the element axis |
| `fields` | a `FieldStore` of `(n_elements, n_timesteps)` or `(n_elements,)` arrays |

Five concrete kinds exist:
`SurfaceDataSource`, `VolumeDataSource`, `PointsDataSource`, `GroupsDataSource`, `ModesDataSource`.

This notebook walks through building, inspecting, and persisting them.

## 1. Build a SurfaceDataSource from numpy arrays

In [1]:
import numpy as np

from cfdmod import (
    SurfaceDataSource,
    TimeAxis,
    Topology,
    ElementMeta,
    MemoryFieldStore,
)

# Two triangles sharing an edge.
vertices = np.array([[0, 0, 0], [1, 0, 0], [0, 1, 0], [1, 1, 0]], dtype=np.float64)
triangles = np.array([[0, 1, 2], [1, 3, 2]], dtype=np.int32)

# A 200-step synthetic pressure series.
rng = np.random.default_rng(0)
pressure = rng.normal(loc=100.0, scale=10.0, size=(2, 200))

ds = SurfaceDataSource(
    time=TimeAxis(initial_time=0.0, timestep_size=0.001, n_timesteps=200),
    topology=Topology.triangles(triangles, vertices),
    elements=ElementMeta(area=np.array([0.5, 0.5])),
    fields=MemoryFieldStore({"pressure": pressure}),
)

print(f"kind={ds.kind}  n_elements={ds.n_elements}  fields={ds.field_names}")
print(
    f"time: {ds.time.n_timesteps} steps from {ds.time.initial_time}s @ dt={ds.time.timestep_size}s"
)

kind=surface  n_elements=2  fields=['pressure']
time: 200 steps from 0.0s @ dt=0.001s


## 2. Inspect the time axis (it's affine, never an array)

In [2]:
# The full array exists on demand, but the DataSource never stores it.
print("first 5 times:", ds.time.times()[:5])
print("time at index 100:", ds.time.time_at(100))
print("index nearest to t=0.123:", ds.time.index_for_time(0.123))

first 5 times: [0.    0.001 0.002 0.003 0.004]
time at index 100: 0.1
index nearest to t=0.123: 123


## 3. Read a field via the FieldStore

The FieldStore is the **only** seam between numpy-in-RAM and h5py-on-disk. Ops never know which backend they're talking to.

In [3]:
# Full read.
p = ds.fields.read("pressure")
print("full shape:", p.shape)

# Time slice (slab read; for an h5 store this lazy-reads from disk).
p_window = ds.fields.read("pressure", time_slice=slice(50, 100))
print("window shape:", p_window.shape)

# Element slice / fancy indexing.
p_one = ds.fields.read("pressure", element_slice=slice(0, 1))
print("one element shape:", p_one.shape)

full shape: (2, 200)
window shape: (2, 50)
one element shape: (1, 200)


## 4. Functional updates (DataSource is frozen)

Every modification returns a fresh `DataSource`. Field arrays are shared by reference unless an op rewrites them, so memory stays flat.

In [4]:
# Add a derived field.
ds2 = ds.with_field("pressure_kPa", ds.fields.read("pressure") / 1000.0)
print("old:", ds.field_names, "-> new:", ds2.field_names)

# Attach a grouping (e.g. body assignment).
from cfdmod import Grouping

g = Grouping(name="body", indices=[0, 0])  # both triangles belong to body 0
ds3 = ds.with_grouping(g)
print("groupings on ds3:", list(ds3.groupings))

old: ['pressure'] -> new: ['pressure', 'pressure_kPa']
groupings on ds3: ['body']


## 5. Persist via the Storage seam

Same code works for in-RAM (tests, notebooks) and h5+xdmf (production).

In [5]:
import tempfile, pathlib
from cfdmod import MemoryStorage, XdmfH5Storage

# In-RAM: trivial dict-keyed storage.
mem = MemoryStorage()
mem.write_data_source("my_run", ds)
round_tripped = mem.read_data_source("my_run")
print("memory round-trip ok:", round_tripped.n_elements == ds.n_elements)

# On-disk: same XDMF + H5 layout the v2 pipeline produces.
with tempfile.TemporaryDirectory() as tmp:
    h5 = XdmfH5Storage(pathlib.Path(tmp))
    h5.write_data_source("my_run", ds)
    files = sorted(pathlib.Path(tmp).iterdir())
    print("on-disk files:", [f.name for f in files])
    back = h5.read_data_source("my_run")
    print(
        "h5 round-trip ok:", np.allclose(back.fields.read("pressure"), ds.fields.read("pressure"))
    )

memory round-trip ok: True


on-disk files: ['my_run.h5', 'my_run.xdmf']
h5 round-trip ok: True


## 6. Other DataSource kinds at a glance

In [6]:
from cfdmod import PointsDataSource, ModesDataSource

# Points: bare probes, no topology connectivity.
probes = np.array([[0, 0, 1.0], [0, 0, 2.0], [0, 0, 5.0]])
u = np.array([[1.0, 1.1], [2.0, 1.9], [4.5, 4.4]])  # (n_probes, n_timesteps)
pts = PointsDataSource(
    time=TimeAxis(initial_time=0.0, timestep_size=0.5, n_timesteps=2),
    topology=Topology.points(probes),
    elements=ElementMeta(position=probes),
    fields=MemoryFieldStore({"u": u}),
)
print(f"PointsDataSource: {pts.n_elements} probes, mean u per probe = {u.mean(axis=1)}")

# Modes: element axis is the mode index.
q = np.zeros((4, 10))  # 4 modes, 10 timesteps
modes = ModesDataSource(
    time=TimeAxis(initial_time=0.0, timestep_size=0.01, n_timesteps=10),
    topology=None,
    elements=ElementMeta(),
    fields=MemoryFieldStore({"q": q}),
)
print(f"ModesDataSource: {modes.n_elements} modes")

PointsDataSource: 3 probes, mean u per probe = [1.05 1.95 4.45]
ModesDataSource: 4 modes


## What's next

- **Tutorial 2 -- Recipes**: build a Cp -> Cf chain with the high-level recipes.
- **Tutorial 3 -- Pipelines**: compose your own pipeline from ops, and run it from a YAML template.
- **Tutorial 4 -- Containers**: organise per-direction / per-case results.